# Recyclability indicator tester

This notebook enables the user test the indicator by playing with the various available input options for each question in the indicator.
It evaluates the product under **priority-constrained Dirichlet weighting distributions** from the final indicator model and returns a Monte Carlo distribution of the recyclability score.

**How to use:**
1. Run all cells , from top to bottom
2. Use the dropdown menus **Section 5** to set the product's sub-indicator values.
3. Click **Evaluate product**.

**Interpretation:** the black line and shaded band show the evaluated product mean score and 95% weighting-uncertainty interval. The Coloured violin plots are simulated product profiles portraying extreme scenarios (Gold Standard and Low Performer) and 2 others used to explore how changes in documentation availability can shift the recyclability score. Regarding the latter two, the Greenwasher represents products with low technical recyclability but high documentation availability scores, while the Hidden Gem represents products with high technical recycling potential but poor documentation availability. It is worth mentioning that these profiles are used for **contextual positioning only**. Given the limitations described below, this notebook can at best provide an approximation of the potential score for a product recyclability and is not intended as a full implementation of the proposed indicator.

**Limitations:**
- This notebook **does not support a full product assessment** as priority part evaluation is not enabled. It considers a single categorical response per criterion rather than a priority part assessment, which would be the ideal implementation of the indicator.
- Recycling Rate is entered as a categorical score and cannot be calculated here from materials efficiencies. This calculation must be done outside this notebook. 

## 1. Imports and indicator configuration

Run this cell once. `OPTION_LABELS` maps every criteria evaluated in the indicator and **normalised score value** to its descriptive text label.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
N_WEIGHT_SAMPLES  = 5000
N_REFERENCE_PRODUCTS = 200

# ── Normalised scoring scales (as described in SI C) ─────────────────────────────
SCALES = {
    'x1x4':   [1.0, 0.8, 0.6, 0.2, 0.0],
    'x5':     [1.0, 0.8, 0.6, 0.4, 0.2, 0.0],
    'x6x7':   [1.0, 0.5, 0.0],
    'x8':     [1.0, 0.8, 0.6, 0.2, 0.0],
    'x9':     [1.0, 0.6, 0.2],
    'x10':    [1.0, 0.8, 0.6, 0.4, 0.2, 0.0],
    'x11':    [1.0, 0.8, 0.6, 0.4, 0.0],
    'binary': [1.0, 0.0],
    'x13':    [1.0, 0.8, 0.6, 0.4, 0.2, 0.0],
    'x15':    [1.0, 0.8, 0.6, 0.4, 0.0],
    'x16':    [1.0, 0.6, 0.2, 0.0],
}

X_TO_SCALE = {
    'X1':  'x1x4', 'X2':  'x1x4', 'X3':  'x1x4', 'X4': 'x1x4',
    'X5':  'x5',   'X6':  'x6x7', 'X7':  'x6x7',
    'X8':  'x8',   'X9':  'x9',   'X10': 'x10',  'X11': 'x11',
    'X12': 'binary', 'X13': 'x13', 'X14': 'binary', 'X15': 'x15',
    'X16': 'x16',
}

# ── Criterion names ──────────────────────────────────────────────────────────
VARIABLE_LABELS = {
    'X1':  'Unique Product Identifier (UPI)',
    'X2':  'Bill of materials',
    'X3':  'Safety manual',
    'X4':  'Disassembly manual',
    'X5':  'How complete is the Bill of material (BoM)',
    'X6':  'Information present on the presence and location of critical raw material (CRM)',
    'X7':  'Information present on the concentration and location of hazardous material in the product',
    'X8':  'List of components/material required to be removed during depollution process',
    'X9':  'Ability to identify the pollutant components/materials (e.g. marking / labelling)',
    'X10': 'Ease of removability of pollutant components/materials',
    'X11': 'Type of tools required to remove the pollutant components',
    'X12': 'Type of fasteners required to remove the pollutant components',
    'X13': 'Ease of removing priority parts',
    'X14': 'Type of tools required to remove priority parts',
    'X15': 'Type of fastener connecting priority parts',
    'X16': 'Recycling rate',
}

# ── Descriptive option labels ────────────────────────────────────────────────
OPTION_LABELS = {
    # Documentation-related criteria (X1–X7)
    'X1': {
        1.0: 'Publicly available ',
        0.8: 'Available to independent recyclers',
        0.6: 'Available to manufacturer-authorized recyclers providers',
        0.2: 'Available to manufacturers only',
        0.0: 'Document is not available',
    },
    'X2': {
        1.0: 'Publicly available ',
        0.8: 'Available to independent recyclers',
        0.6: 'Available to manufacturer-authorized recyclers providers',
        0.2: 'Available to manufacturers only',
        0.0: 'Document is not available',
    },
    'X3': {
        1.0: 'Publicly available ',
        0.8: 'Available to independent recyclers',
        0.6: 'Available to manufacturer-authorized recyclers providers',
        0.2: 'Available to manufacturers only',
        0.0: 'Document is not available',
    },
    'X4': {
        1.0: 'Publicly available ',
        0.8: 'Available to independent recyclers',
        0.6: 'Available to manufacturer-authorized recyclers providers',
        0.2: 'Available to manufacturers only',
        0.0: 'Document is not available',
    },
    'X5': {
        1.0: 'More than 90% of product mass',
        0.8: 'More than 80% of product mass',
        0.6: 'More than 70% of product mass',
        0.4: 'More than 60% of product mass',
        0.2: 'Less than 60% of product mass',
        0.0: 'Data not available ',
    },
    'X6': {
        1.0: 'Completely disclosed to professional recycler',
        0.5: 'Partially disclosed to professional recycler',
        0.0: 'Not disclosed to professional recycler',
    },
    'X7': {
        1.0: 'Completely disclosed to professional recycler',
        0.5: 'Partially disclosed to professional recycler',
        0.0: 'Not disclosed to professional recycler',
    },
    # Depollution-related criteria (X8–X12)
    'X8': {
        1.0: 'Publicly available OR depollution not required',
        0.8: 'Available to independent recyclers',
        0.6: 'Available to manufacturer-authorized recyclers providers',
        0.2: 'Available to manufactures only',
        0.0: 'Data is not available ',
    },
    'X9': {
        1.0: 'Well-marked and identifiable OR depollution not required',
        0.6: 'Not marked but identifiable',
        0.2: 'Not marked and not identifiable ',
    },
    'X10': {
        1.0: '5 ≥ x > 2 steps OR depollution not required',
        0.8: '10 ≥ x > 5 steps',
        0.6: '15 ≥ x > 10 steps',
        0.4: '30≥ x >15 steps',
        0.2: 'x>30 steps',
        0.0: 'Cannot be dismantled',
    },
    'X11': {
        1.0: 'Without tools, or only basic tools OR depollution not required',
        0.8: 'With product group specific tools',
        0.6: 'With commercially available tools',
        0.4: 'With proprietary tools',
        0.0: 'Cannot be dismantled',
    },
    'X12': {
        1.0: 'Removable fastener OR depollution not required',
        0.0: 'Non-removable fastener ',
    },

    # Dismantling-related criteria (X13–X15)
    'X13': {
        1.0: '5 ≥ x > 2 steps',
        0.8: '10 ≥ x > 5 steps',
        0.6: '15 ≥ x > 10 steps',
        0.4: '30≥ x >15 steps',
        0.2: 'x>30 steps',
        0.0: 'Cannot be dismantled',
    },
    'X14': {
        1.0: 'Removable fastener',
        0.0: 'Non-removable fastener',
    },
    'X15': {
        1.0: 'Without tools, or only basic tools',
        0.8: 'With product group specific tools',
        0.6: 'With commercially available tools',
        0.4: 'With proprietary tools',
        0.0: 'Cannot be dismantled',
    },

    # Recycling rate (X16) as categorical score (as described in SI C)
    'X16': {
        1.0: 'More than 60 %',
        0.6: 'Between 50–60 %',
        0.2: 'Below 50 %',
        0.0: 'Cannot be recycled',
    },
}

DOC_COMP       = ['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7']
ARCHETYPE_ORDER = ['Gold Standard', 'Hidden Gem', 'Greenwasher', 'Low Performer']
PALETTE = {
    'Gold Standard': 'gold', 'Hidden Gem': 'steelblue',
    'Greenwasher': 'tomato', 'Low Performer': 'darkred',
}


## 2. Sample the priority-constrained weighting distributions

Constraints follow the final indicator model: w₅ > w₄ > w₁, w₅ > w₄ > w₃, w₅ > w₃ > w₁, w₃ > w₂ (top layer); w₂ > w₃ > w₁, w₂ > w₄ > w₁ (sub-layer A).

In [ ]:
def sample_priority_weights(n_samples, rng):
    """Sample constrained Dirichlet weight sets for all indicator layers."""
    top, layer_a = [], []
    batch_size = max(10_000, n_samples * 10)
    while len(top) < n_samples:
        batch = rng.dirichlet(np.ones(5), size=batch_size)
        valid = batch[
            (batch[:, 4] > batch[:, 3]) & (batch[:, 3] > batch[:, 0]) &
            (batch[:, 3] > batch[:, 2]) & (batch[:, 4] > batch[:, 2]) &
            (batch[:, 2] > batch[:, 1])
        ]
        top.extend(valid[:n_samples - len(top)])
    while len(layer_a) < n_samples:
        batch = rng.dirichlet(np.ones(4), size=batch_size)
        valid = batch[
            (batch[:, 1] > batch[:, 2]) & (batch[:, 1] > batch[:, 3]) &
            (batch[:, 2] > batch[:, 0]) & (batch[:, 3] > batch[:, 0])
        ]
        layer_a.extend(valid[:n_samples - len(layer_a)])
    return {
        'Top': np.asarray(top),
        'A':   np.asarray(layer_a),
        'B':   rng.dirichlet(np.ones(3), size=n_samples),
        'C':   rng.dirichlet(np.ones(5), size=n_samples),
        'D':   rng.dirichlet(np.ones(3), size=n_samples),
    }

weights = sample_priority_weights(N_WEIGHT_SAMPLES, rng)
print(f"Sampled {N_WEIGHT_SAMPLES:,} constrained weight sets.")


## 3. Scoring and product profiles reference functions

In [ ]:
def score_products(product_df, weights):
    """Return a (N_weight_samples, N_products) score matrix."""
    A = weights['A'] @ product_df[['X1', 'X2', 'X3', 'X4']].to_numpy().T
    B = weights['B'] @ product_df[['X5', 'X6', 'X7']].to_numpy().T
    C = weights['C'] @ product_df[['X8', 'X9', 'X10', 'X11', 'X12']].to_numpy().T
    D = weights['D'] @ product_df[['X13', 'X14', 'X15']].to_numpy().T
    E = np.tile(product_df['X16'].to_numpy(), (len(weights['Top']), 1))
    return (
        weights['Top'][:, [0]] * A + weights['Top'][:, [1]] * B +
        weights['Top'][:, [2]] * C + weights['Top'][:, [3]] * D +
        weights['Top'][:, [4]] * E
    )


def select_threshold_values(options, direction, n, rng):
    options = np.asarray(options)
    valid = options[options >= 0.6] if direction == 'high' else options[options <= 0.4]
    return rng.choice(valid if len(valid) else options, size=n)


def map_x16(raw):
    """Apply the normalised 0 / 0.2 / 0.6 / 1.0 X16 scoring rule."""
    return np.where(
        raw <= 0.0, 0.0,
        np.where(raw < 0.5, 0.2, np.where(raw <= 0.6, 0.6, 1.0))
    )


def generate_reference_archetype(archetype, n, rng):
    data = {}
    for i in range(1, 16):
        x, opts = f'X{i}', SCALES[X_TO_SCALE[f'X{i}']]
        is_doc = x in DOC_COMP
        if archetype == 'Gold Standard':
            data[x] = select_threshold_values(opts, 'high', n, rng)
        elif archetype == 'Hidden Gem':
            data[x] = select_threshold_values(opts, 'low' if is_doc else 'high', n, rng)
        elif archetype == 'Greenwasher':
            data[x] = select_threshold_values(opts, 'high' if is_doc else 'low', n, rng)
        else:  # Low Performer
            data[x] = select_threshold_values(opts, 'low', n, rng)
    x16_ranges = {
        'Gold Standard': (0.61, 1.0), 'Hidden Gem': (0.5,  1.0),
        'Greenwasher':   (0.0,  0.49), 'Low Performer': (0.0, 0.49),
    }
    lo, hi = x16_ranges[archetype]
    data['X16'] = map_x16(rng.uniform(lo, hi, size=n))
    return pd.DataFrame(data)


# Pre-compute archetype reference distributions (run once)
reference_scores = {
    arch: score_products(
        generate_reference_archetype(arch, N_REFERENCE_PRODUCTS, rng), weights
    ).ravel()
    for arch in ARCHETYPE_ORDER
}
print('Archetype reference distributions prepared.')


## 4. Select product values

Choose the applicable option for each sub-indicator from the dropdown menus. Criteria are grouped into **documentation** (X1–X7) and **physical / environmental** (X8–X16). After setting all values, click **Evaluate product**.

In [ ]:
def make_dropdown(x):
    """Dropdown with 'score — description' labels for every option."""
    scale_values = sorted(SCALES[X_TO_SCALE[x]], reverse=True)
    labels = OPTION_LABELS.get(x, {})
    options = [
        (f'{v:.1f}  —  {labels.get(v, "")}', v)
        for v in scale_values
    ]
    return widgets.Dropdown(
        options=options,
        value=scale_values[0],
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='560px'),
    )


def criterion_row(x):
    """One labelled row: ID + criterion name on the left, dropdown on the right."""
    label = widgets.HTML(
        value=(
            f'<div style="width:210px; padding-right:10px; line-height:1.5;">'
            f'<b>{x}</b><br>'
            f'<span style="font-size:0.85em; color:#444;">{VARIABLE_LABELS[x]}</span>'
            f'</div>'
        )
    )
    return widgets.HBox([label, make_dropdown(x)])


input_widgets = {f'X{i}': make_dropdown(f'X{i}') for i in range(1, 17)}

def section_header(title):
    return widgets.HTML(
        value=(
            f'<h4 style="margin:14px 0 4px 0; '
            f'border-bottom:1px solid #ccc; padding-bottom:4px;">{title}</h4>'
        )
    )

x10_note = widgets.HTML(
    value=(
        '<div style="margin:2px 0 6px 220px; font-size:0.82em; color:#888; font-style:italic;">'
        '⚠ The available options should be product-specific.'
        '</div>'
    )
)

doc_section = widgets.VBox(
    [section_header('Documentation sub-indicators (X1–X7)')] +
    [criterion_row(f'X{i}') for i in range(1, 8)]
)

phys_rows = []
for i in range(8, 17):
    phys_rows.append(criterion_row(f'X{i}'))
    if i == 10:
        phys_rows.append(x10_note)

phys_section = widgets.VBox(
    [section_header('Physical / environmental sub-indicators (X8–X16)')] + phys_rows
)

evaluate_btn = widgets.Button(
    description='Evaluate product', button_style='primary', icon='play',
    layout=widgets.Layout(margin='12px 8px 0 0'),
)
reset_btn = widgets.Button(
    description='Reset to highest values', icon='refresh',
    layout=widgets.Layout(margin='12px 0 0 0'),
)

panel = widgets.VBox([
    widgets.HTML('<h3>Product sub-indicator values</h3>'),
    doc_section,
    phys_section,
    widgets.HBox([evaluate_btn, reset_btn]),
])
display(panel)


## 5. Evaluate the selected product

In [ ]:
output = widgets.Output()
display(output)


def evaluate_product(_=None):
    values  = {x: w.value for x, w in input_widgets.items()}
    product = pd.DataFrame([values])

    dist   = score_products(product, weights).ravel()
    mean   = dist.mean()
    median = np.median(dist)
    sd     = dist.std(ddof=1)
    p2_5, p97_5 = np.percentile(dist, [2.5, 97.5])

    plot_data = pd.concat(
        [
            pd.DataFrame({
                'Archetype': arch,
                'Recyclability indicator': reference_scores[arch],
            })
            for arch in ARCHETYPE_ORDER
        ],
        ignore_index=True,
    )

    with output:
        clear_output(wait=True)

        # ── Summary statistics ───────────────────────────────────────────────
        print(f"Mean recyclability indicator             : {mean:.3f}")
        print(f"Median recyclability indicator           : {median:.3f}")
        print(f"Std dev (weighting uncertainty)          : {sd:.3f}")
        print(f"95% interval (weighting uncertainty)     : {p2_5:.3f} – {p97_5:.3f}")

        # ── Comparison plot ──────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(11, 6))

        sns.violinplot(
            data=plot_data,
            x='Archetype', y='Recyclability indicator',
            hue='Archetype',
            order=ARCHETYPE_ORDER, hue_order=ARCHETYPE_ORDER,
            palette=PALETTE,
            dodge=False, legend=False,
            inner='quartile', cut=0, linewidth=0.8,
            ax=ax,
        )

        ax.axhline(mean, color='black', linewidth=2,
                   label=f'Selected product mean ({mean:.2f})')
        ax.axhspan(p2_5, p97_5, color='black', alpha=0.10,
                   label=f'95% interval ({p2_5:.2f}–{p97_5:.2f})')

        ax.set_ylim(0, 1)
        ax.set_xlabel('Reference archetype')
        ax.set_ylabel('Recyclability indicator')
        ax.set_title('Selected product relative to archetype reference distributions')
        ax.grid(axis='y', alpha=0.25)
        ax.legend(loc='upper right', frameon=True)
        fig.tight_layout()

        display(fig)
        plt.close(fig)


def reset_to_highest(_=None):
    for w in input_widgets.values():
        w.value = max(o[1] for o in w.options)
    evaluate_product()


evaluate_btn.on_click(evaluate_product)
reset_btn.on_click(reset_to_highest)
evaluate_product()   # show result for default (highest) values


## Notes

- The score distribution reflects **weighting-factor uncertainty** for the user-selected sub-indicator values; it does not account for measurement or classification uncertainty in those values.
- The product profiles violin plots use the same constrained weighting distributions applied to simulated reference products. They provide contextual positioning and should not be interpreted as formal classification thresholds.
- The 95% interval is a **weighting-uncertainty interval**, not a prediction interval for the true recyclability performance.
